# Notebook 12 — Chronological Splitting, Train-Only Scaling & LSTM Sequences

**Project:** Dengue Disease Prediction — Sri Lanka  
**Depends on:** Notebook 11 — Temporal Feature Engineering  
**Forecasting task:** At the end of month **t**, predict district dengue cases in **t + 1**.  
**Outputs:** Train/validation/test metadata, scaled features, six-month LSTM sequences, audit tables, and scaler parameters.

### Research protocol
1. Split by **`target_date`**, not the feature month (`date`).
2. **Train:** target months through 2021-12; **validation:** 2022-01 to 2023-12; **test:** 2024-01 to 2024-12.
3. Fit `StandardScaler` on **training predictors only**. Transform validation/test using that scaler.
4. Build six-month sequences **within each district**, in calendar order. Validation/test sequences may include historical predictor months from earlier partitions (these observations were available by the forecast origin).
5. Keep 2024 test outcomes isolated from feature selection, scaling, model tuning, and final model choice. This notebook performs **no forecasting evaluation**.
6. Preserve **weather-only** and **population-inclusive** experiments separately. Population starts in 2014; never backfill earlier years.

**Important operational assumption:** dengue cases and weather for month *t* are available by the end of month *t*. If official dengue reporting is delayed, this forecasting design must be revised.

## Step 1 — Import libraries and locate the project

Use the `.venv` kernel in VS Code. Required packages: `pandas`, `numpy`, `scikit-learn`, `IPython`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from IPython.display import display

SEED = 42
np.random.seed(SEED)
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data" / "final").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Open D:\\Dengue_Project in VS Code and run from "
        "the project root or its notebooks/ folder."
    )
FINAL_DIR = PROJECT_ROOT / "data" / "final"
MODEL_INPUT_DIR = FINAL_DIR / "model_inputs"
TABLES_DIR = PROJECT_ROOT / "results" / "tables"
MODEL_INPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Model inputs:", MODEL_INPUT_DIR)

Project root: d:\Dengue_Project
Model inputs: d:\Dengue_Project\data\final\model_inputs


## Step 2 — Load Notebook 11 datasets and exact predictor schema

Use the predictor whitelist saved by Notebook 11; never select all numeric columns automatically (that could include the future target).

In [2]:
schema_path = FINAL_DIR / "notebook11_temporal_feature_schema.json"
source_files = {
    "weather_only": FINAL_DIR / "dengue_temporal_features_weather_only_2010_2024.csv",
    "with_population": FINAL_DIR / "dengue_temporal_features_with_population_2014_2024.csv",
}
for path in [schema_path, *source_files.values()]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing Notebook 11 output: {path}. Run Notebook 11 first.")

schema = json.loads(schema_path.read_text(encoding="utf-8"))
TARGET = schema["forecast_target"]
DISTRICT = schema["district_column"]
FEATURE_DATE = schema["feature_date_column"]
TARGET_DATE = schema["target_date_column"]
assert TARGET == "target_dengue_cases_t_plus_1"
assert TARGET_DATE == "target_date"

predictors = {
    "weather_only": list(schema["weather_only_predictors"]),
    "with_population": list(schema["population_predictors"]),
}
raw_data = {
    key: pd.read_csv(path, parse_dates=[FEATURE_DATE, TARGET_DATE])
    for key, path in source_files.items()
}
for variant, frame in raw_data.items():
    print(f"{variant}: {len(frame):,} rows, {len(predictors[variant])} predictors")
    print("  Target coverage:", frame[TARGET_DATE].min().date(), "to", frame[TARGET_DATE].max().date())
    display(frame[[DISTRICT, FEATURE_DATE, TARGET_DATE, TARGET]].head(2))

weather_only: 4,325 rows, 33 predictors
  Target coverage: 2010-08-01 to 2024-12-01


,administrative_district,date,target_date,target_dengue_cases_t_plus_1
0,Ampara,2010-07-01,2010-08-01,20.714286
1,Ampara,2010-08-01,2010-09-01,11.000000


with_population: 3,275 rows, 34 predictors
  Target coverage: 2014-02-01 to 2024-12-01


,administrative_district,date,target_date,target_dengue_cases_t_plus_1
0,Ampara,2014-01-01,2014-02-01,20.000000
1,Ampara,2014-02-01,2014-03-01,19.285714


## Step 3 — Validate the input datasets

Every district must have a unique, uninterrupted monthly series. All predictor values must be finite. A feature row for month *t* must target exactly month *t+1*.

In [3]:
EXPECTED_INPUT_ROWS = {"weather_only": 4325, "with_population": 3275}
EXPECTED_DISTRICTS = 25

for variant, frame in raw_data.items():
    cols = [DISTRICT, FEATURE_DATE, TARGET_DATE, TARGET] + predictors[variant]
    missing_cols = sorted(set(cols) - set(frame.columns))
    if missing_cols:
        raise AssertionError(f"{variant}: missing columns {missing_cols}")
    if len(frame) != EXPECTED_INPUT_ROWS[variant]:
        raise AssertionError(
            f"{variant}: expected {EXPECTED_INPUT_ROWS[variant]} rows; got {len(frame)}. "
            "Check Notebook 11 before proceeding."
        )
    if frame[DISTRICT].nunique() != EXPECTED_DISTRICTS:
        raise AssertionError(f"{variant}: expected 25 districts")
    if frame.duplicated([DISTRICT, FEATURE_DATE]).any():
        raise AssertionError(f"{variant}: duplicate district-feature-month keys")
    if frame.duplicated([DISTRICT, TARGET_DATE]).any():
        raise AssertionError(f"{variant}: duplicate district-target-month keys")
    if frame[cols].isna().any().any():
        raise AssertionError(f"{variant}: missing required input values")
    if not np.isfinite(frame[predictors[variant] + [TARGET]].to_numpy(dtype=float)).all():
        raise AssertionError(f"{variant}: non-finite numeric values")
    if (frame[TARGET] < 0).any():
        raise AssertionError(f"{variant}: negative dengue target")
    if not (frame[TARGET_DATE] == frame[FEATURE_DATE] + pd.offsets.MonthBegin(1)).all():
        raise AssertionError(f"{variant}: target is not the next calendar month")
    for district, group in frame.groupby(DISTRICT):
        sorted_dates = group[FEATURE_DATE].sort_values().reset_index(drop=True)
        expected = pd.date_range(sorted_dates.iloc[0], sorted_dates.iloc[-1], freq="MS")
        if not sorted_dates.equals(pd.Series(expected)):
            raise AssertionError(f"{variant}: missing or repeated months for {district}")
    assert TARGET not in predictors[variant]
    assert TARGET_DATE not in predictors[variant]
    assert FEATURE_DATE not in predictors[variant]
    assert DISTRICT not in predictors[variant]
    assert len(predictors[variant]) == len(set(predictors[variant]))
    print(f"PASS: {variant} input data and predictor whitelist validated")

PASS: weather_only input data and predictor whitelist validated
PASS: with_population input data and predictor whitelist validated


## Step 4 — Define the **target-date** partitions

| Partition | Forecast target date |
|---|---|
| Training | Up to 2021-12 |
| Validation | 2022-01 to 2023-12 |
| Test | 2024-01 to 2024-12 |

**Example:** December 2023 features → January 2024 dengue target, so that row belongs to the **test** partition. Do not randomly shuffle time-series samples.

In [4]:
TRAIN_END = pd.Timestamp("2021-12-01")
VAL_END = pd.Timestamp("2023-12-01")
TEST_END = pd.Timestamp("2024-12-01")
SPLIT_ORDER = ("train", "validation", "test")

def assign_partition(target_dates):
    dates = pd.to_datetime(target_dates)
    labels = np.select(
        [dates <= TRAIN_END, dates <= VAL_END, dates <= TEST_END],
        ["train", "validation", "test"],
        default="outside_protocol",
    )
    return labels

labeled_data = {}
split_summary_rows = []
for variant, frame in raw_data.items():
    frame = frame.copy().sort_values([DISTRICT, FEATURE_DATE]).reset_index(drop=True)
    frame["partition"] = assign_partition(frame[TARGET_DATE])
    if frame["partition"].eq("outside_protocol").any():
        raise AssertionError(f"{variant}: target date beyond 2024-12")
    labeled_data[variant] = frame
    for partition in SPLIT_ORDER:
        part = frame.loc[frame["partition"].eq(partition)]
        split_summary_rows.append({
            "dataset": variant, "partition": partition, "rows": len(part),
            "districts": part[DISTRICT].nunique(),
            "first_target_date": part[TARGET_DATE].min(),
            "last_target_date": part[TARGET_DATE].max(),
        })
split_summary = pd.DataFrame(split_summary_rows)
display(split_summary)
EXPECTED_SPLITS = {
    "weather_only": {"train": 3425, "validation": 600, "test": 300},
    "with_population": {"train": 2375, "validation": 600, "test": 300},
}
for variant in labeled_data:
    for partition in SPLIT_ORDER:
        count = len(labeled_data[variant].query("partition == @partition"))
        assert count == EXPECTED_SPLITS[variant][partition], (variant, partition, count)
print("PASS: chronological target-date splits match Notebook 11 coverage")

,dataset,partition,rows,districts,first_target_date,last_target_date
0,weather_only,train,3425,25,2010-08-01,2021-12-01
1,weather_only,validation,600,25,2022-01-01,2023-12-01
2,weather_only,test,300,25,2024-01-01,2024-12-01
3,with_population,train,2375,25,2014-02-01,2021-12-01
4,with_population,validation,600,25,2022-01-01,2023-12-01
5,with_population,test,300,25,2024-01-01,2024-12-01


PASS: chronological target-date splits match Notebook 11 coverage


## Step 5 — Fit a scaler on training predictors **only**

`StandardScaler` uses each feature’s training mean and standard deviation. The fitted parameters are **never updated** using validation or test rows. This is critical for preventing preprocessing leakage.

In [5]:
scalers = {}
scaled_data = {}
scaler_rows = []

for variant, frame in labeled_data.items():
    names = predictors[variant]
    train_mask = frame["partition"].eq("train")
    scaler = StandardScaler()
    scaler.fit(frame.loc[train_mask, names].to_numpy(dtype=np.float64))
    transformed = scaler.transform(frame[names].to_numpy(dtype=np.float64))
    if not np.isfinite(transformed).all():
        raise AssertionError(f"{variant}: scaling produced non-finite values")
    scaled = frame.copy()
    scaled.loc[:, names] = transformed
    scalers[variant] = scaler
    scaled_data[variant] = scaled
    for j, name in enumerate(names):
        scaler_rows.append({
            "dataset": variant, "feature": name,
            "train_mean": float(scaler.mean_[j]),
            "train_scale": float(scaler.scale_[j]),
            "train_variance": float(scaler.var_[j]),
            "training_rows_used": int(train_mask.sum()),
        })
    print(f"{variant}: fitted scaler on {train_mask.sum():,} training rows; transformed {len(frame):,} rows")

scaler_audit = pd.DataFrame(scaler_rows)
display(scaler_audit.head(8))

weather_only: fitted scaler on 3,425 training rows; transformed 4,325 rows
with_population: fitted scaler on 2,375 training rows; transformed 3,275 rows


,dataset,feature,train_mean,train_scale,train_variance,training_rows_used
0,weather_only,dengue_cases,157.938936,404.396884,163536.839650,3425
1,weather_only,dengue_cases_lag1,158.239666,404.477973,163602.430306,3425
2,weather_only,dengue_cases_lag2,158.261564,404.441826,163573.190485,3425
3,weather_only,dengue_cases_lag3,158.379479,404.398272,163537.962345,3425
4,weather_only,dengue_cases_lag6,159.586861,404.303517,163461.334068,3425
5,weather_only,dengue_cases_roll3_mean,158.146722,366.167410,134078.572159,3425
6,weather_only,dengue_cases_roll6_mean,158.425895,328.687240,108035.301652,3425
7,weather_only,temperature_mean_c,26.163148,2.558994,6.548452,3425


## Step 6 — Check train-only scaling and inverse reconstruction

The training feature means should be approximately zero, except features with zero variance (which become zero). Inverting the scaler must recover the original values.

In [6]:
for variant, scaled in scaled_data.items():
    names = predictors[variant]
    train_mask = scaled["partition"].eq("train")
    train_values = scaled.loc[train_mask, names].to_numpy(dtype=float)
    if not np.allclose(train_values.mean(axis=0), 0.0, atol=1e-9):
        raise AssertionError(f"{variant}: training means not centered")
    recovered = scalers[variant].inverse_transform(scaled[names].to_numpy(dtype=float))
    original = labeled_data[variant][names].to_numpy(dtype=float)
    if not np.allclose(recovered, original, atol=1e-7, rtol=1e-10):
        raise AssertionError(f"{variant}: inverse scaling mismatch")
    if not np.allclose(scalers[variant].mean_, original[train_mask.to_numpy()].mean(axis=0)):
        raise AssertionError(f"{variant}: scaler means do not match train-only means")
    print(f"PASS: {variant} train-only fit and inverse transform")

PASS: weather_only train-only fit and inverse transform
PASS: with_population train-only fit and inverse transform


## Step 7 — Construct six-month LSTM sequences

For a forecast of dengue in month **t+1**, the model receives six consecutive feature rows ending in month **t**:

\[
X = [x_{t-5}, x_{t-4}, x_{t-3}, x_{t-2}, x_{t-1}, x_t],\qquad y = \mathrm{cases}_{t+1}.
\]

- Sequence shape: `(number_of_samples, 6, number_of_predictors)`.
- **No district mixing**: each sequence comes from exactly one administrative district.
- **No gaps**: the six feature months must be consecutive.
- **No leakage**: the latest input month is strictly before the target month.
- **Historical context allowed**: January 2022 validation sequences may include feature rows from 2021; January 2024 test sequences may include features from 2023. Those are observed history, not future targets.
- Earlier rows are used **only as input features**, not as additional labels in a sequence.

Note: Notebook 11 already includes lag and rolling features, so this six-month LSTM sequence adds an additional temporal context window. Keep this design fixed when comparing models.

In [7]:
SEQUENCE_LENGTH = 6

def make_sequences(frame, feature_names, sequence_length=6):
    xs, ys, ys_log, records = [], [], [], []
    for district, group in frame.groupby(DISTRICT, sort=True):
        group = group.sort_values(FEATURE_DATE).reset_index(drop=True)
        values = group[feature_names].to_numpy(dtype=np.float32)
        for end in range(sequence_length - 1, len(group)):
            start = end - sequence_length + 1
            window = group.iloc[start:end + 1]
            expected_dates = pd.date_range(
                window[FEATURE_DATE].iloc[0], periods=sequence_length, freq="MS"
            )
            if not np.array_equal(window[FEATURE_DATE].to_numpy(dtype="datetime64[ns]"),
                                  expected_dates.to_numpy(dtype="datetime64[ns]")):
                raise AssertionError(f"Gap in sequence for {district}")
            last = window.iloc[-1]
            if last[FEATURE_DATE] >= last[TARGET_DATE]:
                raise AssertionError("Input month must precede target month")
            xs.append(values[start:end + 1])
            y_value = float(last[TARGET])
            ys.append(y_value)
            ys_log.append(np.log1p(y_value))
            records.append({
                "administrative_district": district,
                "sequence_start_date": window[FEATURE_DATE].iloc[0],
                "feature_date": last[FEATURE_DATE],
                "target_date": last[TARGET_DATE],
                "partition": last["partition"],
                "target_dengue_cases": y_value,
            })
    return (
        np.stack(xs).astype(np.float32),
        np.asarray(ys, dtype=np.float32),
        np.asarray(ys_log, dtype=np.float32),
        pd.DataFrame.from_records(records),
    )

sequence_data = {}
sequence_summary_rows = []
for variant, frame in scaled_data.items():
    X, y, y_log, metadata = make_sequences(frame, predictors[variant], SEQUENCE_LENGTH)
    assert len(X) == len(y) == len(y_log) == len(metadata)
    assert X.shape[1:] == (SEQUENCE_LENGTH, len(predictors[variant]))
    assert np.isfinite(X).all() and np.isfinite(y).all()
    sequence_data[variant] = (X, y, y_log, metadata)
    for partition in SPLIT_ORDER:
        portion = metadata.loc[metadata["partition"].eq(partition)]
        sequence_summary_rows.append({
            "dataset": variant, "partition": partition,
            "sequences": len(portion),
            "districts": portion["administrative_district"].nunique(),
            "first_target_date": portion["target_date"].min(),
            "last_target_date": portion["target_date"].max(),
        })
    print(f"{variant}: X={X.shape}, y={y.shape}, y_log1p={y_log.shape}")
sequence_summary = pd.DataFrame(sequence_summary_rows)
display(sequence_summary)
EXPECTED_SEQUENCES = {
    "weather_only": {"train": 3300, "validation": 600, "test": 300},
    "with_population": {"train": 2250, "validation": 600, "test": 300},
}
for variant in sequence_data:
    for partition in SPLIT_ORDER:
        n = len(sequence_data[variant][3].query("partition == @partition"))
        assert n == EXPECTED_SEQUENCES[variant][partition], (variant, partition, n)
print("PASS: expected six-month sequence counts")

weather_only: X=(4200, 6, 33), y=(4200,), y_log1p=(4200,)
with_population: X=(3150, 6, 34), y=(3150,), y_log1p=(3150,)


,dataset,partition,sequences,districts,first_target_date,last_target_date
0,weather_only,train,3300,25,2011-01-01,2021-12-01
1,weather_only,validation,600,25,2022-01-01,2023-12-01
2,weather_only,test,300,25,2024-01-01,2024-12-01
3,with_population,train,2250,25,2014-07-01,2021-12-01
4,with_population,validation,600,25,2022-01-01,2023-12-01
5,with_population,test,300,25,2024-01-01,2024-12-01


PASS: expected six-month sequence counts


## Step 8 — Validate split boundaries and sequence history

Check that **all** test targets are in 2024, that validation/test sequences can use prior-period historical inputs without crossing into their **future**, and that no sample spans multiple districts. Note that this notebook never trains on 2024 target labels.

In [8]:
for variant, (X, y, y_log, metadata) in sequence_data.items():
    assert not metadata.duplicated(["administrative_district", "target_date"]).any()
    assert (metadata["target_date"] == metadata["feature_date"] + pd.offsets.MonthBegin(1)).all()
    assert (metadata["sequence_start_date"] == metadata["feature_date"] - pd.DateOffset(months=SEQUENCE_LENGTH-1)).all()
    assert np.allclose(np.expm1(y_log.astype(np.float64)), y.astype(np.float64), atol=0.01, rtol=1e-5)
    for partition in SPLIT_ORDER:
        m = metadata.loc[metadata["partition"].eq(partition)]
        assert len(m) == EXPECTED_SEQUENCES[variant][partition]
        assert m["administrative_district"].nunique() == 25
        if partition == "train":
            assert m["target_date"].max() <= TRAIN_END
        elif partition == "validation":
            assert m["target_date"].min() > TRAIN_END
            assert m["target_date"].max() <= VAL_END
        else:
            assert m["target_date"].min() > VAL_END
            assert m["target_date"].max() == TEST_END
    print(f"PASS: {variant} sequence dates, labels, partitions, and district separation")

PASS: weather_only sequence dates, labels, partitions, and district separation
PASS: with_population sequence dates, labels, partitions, and district separation


## Step 9 — Save model-ready arrays, row metadata and scaler parameters

Files go into `data/final/model_inputs/` and audit tables into `results/tables/`.

For each variant, this notebook saves:
- `*_train.npz`, `*_validation.npz`, `*_test.npz`: `X`, `y_raw`, `y_log1p`, and `feature_names`.
- `*_train_metadata.csv`, etc.: district, sequence start, feature month, target month, target cases.
- `*_scaler.json`: **training-only** mean, scale and variance; feature order; split rules; sequence length.
- `*_scaled_rows.csv`: scaled tabular rows, useful for inspecting feature values.

**Notebook 13 should train using `y_log1p` (not both labels).** Convert predictions back to the original dengue-case scale with `np.expm1` before computing MAE/RMSE. Do not apply `np.expm1` to `y_raw`. Never compute final 2024 test metrics until model selection is complete.

In [9]:
saved_paths = []
for variant, frame in scaled_data.items():
    feature_names = predictors[variant]
    scaler = scalers[variant]
    scaler_info = {
        "dataset": variant,
        "method": "sklearn.preprocessing.StandardScaler",
        "fit_partition": "train_only",
        "feature_names": feature_names,
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
        "variance": scaler.var_.tolist(),
        "n_training_rows": int(frame["partition"].eq("train").sum()),
        "sequence_length": SEQUENCE_LENGTH,
        "forecast_target": TARGET,
        "target_transform_for_training": "log1p",
        "prediction_inverse_transform": "expm1",
        "split_basis": "target_date",
        "train_target_end": str(TRAIN_END.date()),
        "validation_target_end": str(VAL_END.date()),
        "test_target_end": str(TEST_END.date()),
        "district_encoding": "not_encoded",
        "note": "No scaler or model was fitted on validation/test rows or labels.",
    }
    scaler_path = MODEL_INPUT_DIR / f"{variant}_scaler.json"
    scaler_path.write_text(json.dumps(scaler_info, indent=2), encoding="utf-8")
    saved_paths.append(scaler_path)
    tabular_path = MODEL_INPUT_DIR / f"{variant}_scaled_rows.csv"
    frame[[DISTRICT, FEATURE_DATE, TARGET_DATE, "partition"] + feature_names + [TARGET]].to_csv(
        tabular_path, index=False
    )
    saved_paths.append(tabular_path)

    X, y, y_log, metadata = sequence_data[variant]
    for partition in SPLIT_ORDER:
        mask = metadata["partition"].eq(partition).to_numpy()
        npz_path = MODEL_INPUT_DIR / f"{variant}_{partition}.npz"
        np.savez_compressed(
            npz_path,
            X=X[mask], y_raw=y[mask], y_log1p=y_log[mask],
            feature_names=np.asarray(feature_names, dtype=str),
        )
        saved_paths.append(npz_path)
        meta_path = MODEL_INPUT_DIR / f"{variant}_{partition}_metadata.csv"
        metadata.loc[mask].to_csv(meta_path, index=False)
        saved_paths.append(meta_path)
        print(f"{variant}/{partition}: {int(mask.sum()):,} sequences saved")

for filename, table in [
    ("notebook12_split_summary.csv", split_summary),
    ("notebook12_sequence_summary.csv", sequence_summary),
    ("notebook12_train_scaler_audit.csv", scaler_audit),
]:
    path = TABLES_DIR / filename
    table.to_csv(path, index=False)
    saved_paths.append(path)

print(f"Saved {len(saved_paths)} files.")
for path in saved_paths:
    print(" -", path.relative_to(PROJECT_ROOT))

weather_only/train: 3,300 sequences saved
weather_only/validation: 600 sequences saved
weather_only/test: 300 sequences saved
with_population/train: 2,250 sequences saved
with_population/validation: 600 sequences saved
with_population/test: 300 sequences saved
Saved 19 files.
 - data\final\model_inputs\weather_only_scaler.json
 - data\final\model_inputs\weather_only_scaled_rows.csv
 - data\final\model_inputs\weather_only_train.npz
 - data\final\model_inputs\weather_only_train_metadata.csv
 - data\final\model_inputs\weather_only_validation.npz
 - data\final\model_inputs\weather_only_validation_metadata.csv
 - data\final\model_inputs\weather_only_test.npz
 - data\final\model_inputs\weather_only_test_metadata.csv
 - data\final\model_inputs\with_population_scaler.json
 - data\final\model_inputs\with_population_scaled_rows.csv
 - data\final\model_inputs\with_population_train.npz
 - data\final\model_inputs\with_population_train_metadata.csv
 - data\final\model_inputs\with_population_validati

## Step 10 — Reload saved arrays and perform final integrity checks

A successful run ends with `ALL NOTEBOOK 12 CHECKS PASSED`.

In [10]:
for variant in predictors:
    feature_names = predictors[variant]
    for partition in SPLIT_ORDER:
        path = MODEL_INPUT_DIR / f"{variant}_{partition}.npz"
        metadata_path = MODEL_INPUT_DIR / f"{variant}_{partition}_metadata.csv"
        with np.load(path, allow_pickle=False) as loaded:
            assert loaded["X"].shape == (
                EXPECTED_SEQUENCES[variant][partition], SEQUENCE_LENGTH, len(feature_names)
            )
            assert loaded["y_raw"].shape == (EXPECTED_SEQUENCES[variant][partition],)
            assert loaded["y_log1p"].shape == loaded["y_raw"].shape
            assert loaded["feature_names"].tolist() == feature_names
            assert np.isfinite(loaded["X"]).all()
        assert metadata_path.is_file()
        assert len(pd.read_csv(metadata_path)) == EXPECTED_SEQUENCES[variant][partition]
assert all(path.is_file() for path in saved_paths)
print("ALL NOTEBOOK 12 CHECKS PASSED")
print("Next: Notebook 13 — Fixed LSTM baseline (train/validation first; test only after finalizing the model)")

ALL NOTEBOOK 12 CHECKS PASSED
Next: Notebook 13 — Fixed LSTM baseline (train/validation first; test only after finalizing the model)


## How to interpret these outputs

- **Split summary:** counts of target-month rows assigned to train, validation and test. These are not model scores.
- **Scaler audit:** the mean and scale learned **only** from the training period. No validation/test fitting occurs.
- **Sequence summary:** the number of usable six-month LSTM windows. The first five rows of each district cannot form a six-month sequence, so training counts are smaller than tabular counts.
- **`X` in `.npz`:** standardized predictors with shape `(samples, 6, features)`.
- **`y_log1p`:** suggested neural-network training target. At evaluation time, convert predicted log-cases with `np.expm1`; clip negative reconstructed case counts to zero if needed.
- **`y_raw`:** original next-month dengue cases, retained for reporting MAE/RMSE on the correct scale.
- **Population variant:** training starts later than the weather-only variant. To compare their model quality, use the same test targets and consider matched-period training ablations.

### Important limitations

1. All target labels are present in the saved `.npz` files, including test labels, for reproducible **final evaluation**. Notebook 13 must not use `*_test.npz` for training, early stopping, hyperparameter selection, or repeated trial-and-error.
2. The six-month sequence design reuses Notebook 11 lagged/rolling predictors, which already summarize historical months. This is a deliberate modeling choice, not a proof of optimal architecture.
3. Population values are annual values repeated monthly, not new monthly measurements. Check whether the published annual population estimate would actually have been available at the forecast origin; if not, use the weather-only variant for strict operational forecasting.
4. `StandardScaler` is sensitive to extreme values. Alternatives may be evaluated using training/validation only, but 2024 must remain untouched.
5. This notebook does **not** perform imputation. Notebook 08 was an artificial-missingness benchmark; the prepared internal weather series had no ordinary missing values.

**Next:** Notebook 13 will load `weather_only_train.npz` and `weather_only_validation.npz`, establish a fixed LSTM baseline, and save the chosen model without looking at test performance.